# Dados Antes e Depois do Processamento

Este notebook compara o `Telco-Customer-Churn.csv` em três momentos:

1. Dados brutos, como chegam no CSV.
2. Dados após limpeza semântica e engenharia de features.
3. Matriz final numérica usada pelos modelos Scikit-Learn e pela MLP em PyTorch.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tech_challenge_churn.config import BASE_CATEGORICAL_FEATURES, BASE_NUMERIC_FEATURES, INTERNET_DEPENDENT_COLUMNS
from tech_challenge_churn.data.load import read_raw_data, split_features_target
from tech_challenge_churn.data.schema import validate_telco_schema
from tech_challenge_churn.features.build import add_telco_features, build_feature_pipeline

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)

DATA_PATH = PROJECT_ROOT / "Telco-Customer-Churn.csv"

## 1. Dados Brutos

Nesta etapa carregamos o CSV original e validamos o schema esperado, mas ainda não removemos colunas, não convertemos `TotalCharges` e não codificamos variáveis categóricas.

In [ ]:
raw_data = validate_telco_schema(read_raw_data(DATA_PATH))

print(f"Linhas: {raw_data.shape[0]:,}".replace(",", "."))
print(f"Colunas: {raw_data.shape[1]}")
display(raw_data.head())

In [ ]:
raw_profile = pd.DataFrame(
    {
        "tipo_original": raw_data.dtypes.astype(str),
        "nulos_pandas": raw_data.isna().sum(),
        "valores_unicos": raw_data.nunique(dropna=False),
        "exemplo_1": raw_data.iloc[0].astype(str),
    }
)
display(raw_profile)

In [ ]:
categorical_columns = [column for column in raw_data.select_dtypes(include="object").columns if column != "customerID"]
categorical_domains = []

for column in categorical_columns:
    values = sorted(raw_data[column].dropna().astype(str).unique().tolist())
    categorical_domains.append(
        {
            "coluna": column,
            "quantidade_categorias": len(values),
            "valores_observados": values,
        }
    )

display(pd.DataFrame(categorical_domains))

## 2. Separação de Features e Alvo

`customerID` é removida das features e `Churn` é convertido para alvo binário: `Yes = 1`, `No = 0`.

In [ ]:
features, target = split_features_target(raw_data)

print(f"Features antes do processamento: {features.shape}")
display(features.head())
display(target.value_counts().rename(index={0: "No", 1: "Yes"}).rename("quantidade"))

## 3. Depois da Limpeza Semântica e Engenharia de Features

Aqui o pipeline ainda está em formato de DataFrame. Esta etapa normaliza categorias redundantes, converte `TotalCharges` para número, trata casos de `tenure=0` e cria variáveis derivadas de gasto, contrato e serviços.

In [ ]:
engineered = add_telco_features(features)
created_columns = [column for column in engineered.columns if column not in features.columns]

print(f"Colunas antes da engenharia: {features.shape[1]}")
print(f"Colunas depois da engenharia: {engineered.shape[1]}")
print(f"Features criadas: {len(created_columns)}")
display(pd.Series(created_columns, name="feature_criada"))

In [ ]:
normalization_rows = []
columns_with_semantic_normalization = ["MultipleLines", *INTERNET_DEPENDENT_COLUMNS]

for column in columns_with_semantic_normalization:
    before_values = sorted(features[column].dropna().astype(str).unique().tolist())
    after_values = sorted(engineered[column].dropna().astype(str).unique().tolist())
    normalization_rows.append(
        {
            "coluna": column,
            "valores_antes": before_values,
            "valores_depois": after_values,
        }
    )

display(pd.DataFrame(normalization_rows))

In [ ]:
comparison_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "avg_monthly_spend",
    "charges_delta",
    "total_to_monthly_ratio",
    "tenure_bucket",
    "num_services",
    "num_protection_services",
    "has_internet_service",
    "fiber_without_security",
    "electronic_check_month_to_month",
]

display(engineered[comparison_columns].head(10))

## 4. Matriz Final Depois do Pré-Processamento

Nesta etapa aplicamos o `build_feature_pipeline`, que executa a engenharia de features e depois transforma tudo em números:

- Numéricas: imputação por mediana e `StandardScaler`.
- Categóricas: imputação por moda e `OneHotEncoder`.
- Binárias categóricas: `OneHotEncoder(drop="if_binary")`, mantendo apenas uma coluna.

In [ ]:
feature_pipeline = build_feature_pipeline()
processed_array = feature_pipeline.fit_transform(features, target)
feature_names = feature_pipeline.named_steps["preprocessor"].get_feature_names_out()
processed_data = pd.DataFrame(processed_array, columns=feature_names, index=features.index)

print(f"Formato bruto completo: {raw_data.shape}")
print(f"Features antes do processamento: {features.shape}")
print(f"DataFrame com features criadas: {engineered.shape}")
print(f"Matriz final para modelagem: {processed_data.shape}")
display(processed_data.head())

In [ ]:
numeric_feature_names = [name for name in processed_data.columns if name.startswith("numeric__")]
categorical_feature_names = [name for name in processed_data.columns if name.startswith("categorical__")]

print(f"Features numéricas após escala: {len(numeric_feature_names)}")
print(f"Features categóricas após one-hot: {len(categorical_feature_names)}")

numeric_summary = processed_data[numeric_feature_names].agg(["mean", "std", "min", "max"]).T
display(numeric_summary.head(25).round(4))

In [ ]:
display(pd.Series(numeric_feature_names, name="features_numericas_finais"))
display(pd.Series(categorical_feature_names, name="features_categoricas_one_hot").head(80))

## 5. Leitura Prática

A base original tem colunas legíveis para análise humana. Depois do pipeline, o modelo recebe uma matriz totalmente numérica, com variáveis contínuas padronizadas e variáveis categóricas expandidas por one-hot. Essa matriz final é a mesma base de entrada usada pelos baselines Scikit-Learn, pelos modelos tunados e pela MLP.

## 6. Resultado da Ablação de Features

Depois da rodada de ablação, usamos o RandomForest campeão como modelo fixo e alteramos apenas o conjunto de atributos. O objetivo foi identificar colunas ou grupos de colunas que poderiam ser removidos ou resumidos sem piorar o F1.

In [ ]:
ablation_path = PROJECT_ROOT / "reports" / "feature_ablation" / "comparison.csv"
ablation = pd.read_csv(ablation_path)

ablation_columns = [
    "model",
    "final_feature_count",
    "f1_mean",
    "delta_f1_mean",
    "optimal_f1_mean",
    "delta_optimal_f1_mean",
    "pr_auc_mean",
    "delta_pr_auc_mean",
    "removed_final_features",
    "is_non_degrading_f1",
]

display(
    ablation[ablation_columns]
    .sort_values(["f1_mean", "optimal_f1_mean", "pr_auc_mean"], ascending=False)
    .round(4)
)

In [ ]:
reference = ablation.query("model == 'full_current'").iloc[0]
best_f1 = ablation.sort_values("f1_mean", ascending=False).iloc[0]
relationship_summary = ablation.query("model == 'relationship_summarized'").iloc[0]

print("Referência atual")
print(f"- Features finais: {int(reference['final_feature_count'])}")
print(f"- F1 médio: {reference['f1_mean']:.4f}")
print(f"- PR-AUC média: {reference['pr_auc_mean']:.4f}")

print("\nMelhor ablação por F1")
print(f"- Feature set: {best_f1['model']}")
print(f"- Features finais: {int(best_f1['final_feature_count'])}")
print(f"- F1 médio: {best_f1['f1_mean']:.4f}")
print(f"- Delta F1: {best_f1['delta_f1_mean']:+.4f}")

print("\nSumarização de relacionamento")
print("- Substitui Partner e Dependents por has_family_context")
print(f"- F1 médio: {relationship_summary['f1_mean']:.4f}")
print(f"- Delta F1: {relationship_summary['delta_f1_mean']:+.4f}")

### Recomendação após a ablação

- `gender` pode ser removida: o F1 médio passou de `0.6395` para `0.6402`, reduzindo uma feature e diminuindo risco ético.
- `Partner` e `Dependents` podem ser resumidas em `has_family_context`: o F1 médio ficou em `0.6399`, sem piora contra a referência.
- `StreamingTV` e `StreamingMovies` podem ser resumidas em `streaming_bundle`, mas houve pequena perda de F1; usar apenas se simplicidade for mais importante que o máximo desempenho.
- Serviços de proteção individuais ainda carregam sinal útil; resumir tudo em contagem reduziu F1 e PR-AUC.

Relatório completo: `docs/feature_ablation_report.md`.